In [6]:
pip install ucimlrepo 

  Using cached ucimlrepo-0.0.7-py3-none-any.whl.metadata (5.5 kB)
Using cached ucimlrepo-0.0.7-py3-none-any.whl (8.0 kB)
Note: you may need to restart the kernel to use updated packages.


In [11]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

df = pd.concat([X, y], axis=1)

print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

Shape: (48842, 15)
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0        

In [12]:
# Shape and data types
print("Rows, columns:", df.shape)
print("\nData types:")
print(df.dtypes)

# First records
display(df.head())

# Numeric summary
display(df.describe(include="all").T)

# Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})
display(missing_report)

# Duplicate records
print("Duplicate rows:", df.duplicated().sum())

# Category inspection
for col in df.select_dtypes(include="object"):
    print(f"\n{col}:")


Rows, columns: (48842, 15)

Data types:
age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
income            object
dtype: object


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,48842.0,NaN,NaN,NaN,38.643585,13.71051,17.0,28.0,37.0,48.0,90.0
workclass,47879,9,Private,33906,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fnlwgt,48842.0,NaN,NaN,NaN,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education,48842,16,HS-grad,15784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education-num,48842.0,NaN,NaN,NaN,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
marital-status,48842,7,Married-civ-spouse,22379,NaN,NaN,NaN,NaN,NaN,NaN,NaN
occupation,47876,15,Prof-specialty,6172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
relationship,48842,6,Husband,19716,NaN,NaN,NaN,NaN,NaN,NaN,NaN
race,48842,5,White,41762,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,48842,2,Male,32650,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,missing_count,missing_percent
occupation,966,1.98
workclass,963,1.97
native-country,274,0.56
age,0,0.00
fnlwgt,0,0.00
education,0,0.00
education-num,0,0.00
marital-status,0,0.00
relationship,0,0.00
race,0,0.00


Duplicate rows: 29

workclass:

education:

marital-status:

occupation:

relationship:

race:

sex:

native-country:

income:


In [13]:
initial_shape = df.shape
initial_missing = df.isna().sum()
initial_duplicates = df.duplicated().sum()

quality_baseline = {
    "rows": initial_shape[0],
    "columns": initial_shape[1],
    "total_missing_cells": int(initial_missing.sum()),
    "duplicate_rows": int(initial_duplicates)
}
print(quality_baseline)


{'rows': 48842, 'columns': 15, 'total_missing_cells': 2203, 'duplicate_rows': 29}


In [14]:
# Treat question marks as missing
df = df.replace(["?", " ?"], np.nan)

print(df.isna().sum().sort_values(ascending=False))


occupation        2809
workclass         2799
native-country     857
age                  0
fnlwgt               0
education            0
education-num        0
marital-status       0
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
income               0
dtype: int64


In [15]:
cat_cols = df.select_dtypes(include="object").columns
num_cols = df.select_dtypes(exclude="object").columns

# Categorical: mode
for col in cat_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

# Numeric: median
for col in num_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("Remaining missing cells:", int(df.isna().sum().sum()))


Remaining missing cells: 0


In [16]:
# Remove leading/trailing whitespace from object columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

# Re-check category counts
for col in ["workclass", "education", "occupation", "native-country"]:
    print(col, df[col].nunique(), "unique values")


workclass 8 unique values
education 16 unique values
occupation 14 unique values
native-country 41 unique values


In [17]:
# Basic range checks
checks = {
    "age_nonpositive": (df["age"] <= 0).sum(),
    "education_num_invalid": (~df["education-num"].between(1, 16)).sum(),
    "hours_nonpositive": (df["hours-per-week"] <= 0).sum(),
    "capital_gain_negative": (df["capital-gain"] < 0).sum(),
    "capital_loss_negative": (df["capital-loss"] < 0).sum(),
}

for name, count in checks.items():
    print(name, ":", int(count))


age_nonpositive : 0
education_num_invalid : 0
hours_nonpositive : 0
capital_gain_negative : 0
capital_loss_negative : 0


In [18]:
allowed_workclass = {
    "Private", "Self-emp-not-inc", "Self-emp-inc", "Federal-gov",
    "Local-gov", "State-gov", "Without-pay", "Never-worked"
}

unexpected_workclass = set(df["workclass"].dropna().unique()) - allowed_workclass
print("Unexpected workclass values:", unexpected_workclass)


Unexpected workclass values: set()


In [19]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows before removal:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after duplicate removal:", df.shape)


Duplicate rows before removal: 29
Shape after duplicate removal: (48813, 15)


In [20]:
numeric_for_outlier_check = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]

def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_summary = []

for col in numeric_for_outlier_check:
    lower, upper = iqr_bounds(df[col])
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_summary.append({
        "feature": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
        "outlier_percent": round(mask.mean() * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_summary)
display(outlier_summary)


,feature,lower_bound,upper_bound,outlier_count,outlier_percent
0,age,-2.0,78.0,215,0.44
1,fnlwgt,-62542.5,417717.5,1453,2.98
2,education-num,4.5,16.5,1789,3.67
3,capital-gain,0.0,0.0,4035,8.27
4,capital-loss,0.0,0.0,2282,4.67
5,hours-per-week,32.5,52.5,13489,27.63


In [22]:
# Optional winsorization function
def winsorize_by_percentile(series, low=0.01, high=0.99):
    lo = series.quantile(low)
    hi = series.quantile(high)
    return series.clip(lower=lo, upper=hi)

# Apply only after a documented decision that winsorization is appropriate
for col in ["fnlwgt", "capital-gain", "capital-loss"]:
    df[col] = winsorize_by_percentile(df[col])


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target_col = "income"

X = df.drop(columns=[target_col])
y = df[target_col].str.replace(".", "", regex=False).str.strip()

categorical_cols = X.select_dtypes(include=["object", "string"]).columns
numeric_cols = X.select_dtypes(exclude=["object", "string"]).columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

X_processed = preprocessor.fit_transform(X)

print("Processed feature matrix shape:", X_processed.shape)


Processed feature matrix shape: (48813, 105)


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [26]:
# Final quality checks
print("Final shape:", df.shape)
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

# Numeric validity
assert (df["age"] > 0).all()
assert (df["education-num"].between(1, 16)).all()
assert (df["hours-per-week"] > 0).all()

# Confirm target labels
print("Target classes:", sorted(df[target_col].unique()))


Final shape: (48813, 15)
Missing cells: 0
Duplicate rows: 3
Target classes: ['<=50K', '<=50K.', '>50K', '>50K.']


In [29]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# -----------------------------
# 1. Acquire
# -----------------------------
adult = fetch_ucirepo(id=2)
X = adult.data.features.copy()
y = adult.data.targets.copy()
df = pd.concat([X, y], axis=1)

raw_df = df.copy()
print("Raw shape:", df.shape)

# -----------------------------
# 2. Standardize missing markers
# -----------------------------
df = df.replace(["?", " ?"], np.nan)

# -----------------------------
# 3. Normalize strings
# -----------------------------
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# -----------------------------
# 4. Audit duplicates
# -----------------------------
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# -----------------------------
# 5. Validate ranges
# -----------------------------
print("Invalid ages:", (df["age"] <= 0).sum())
print("Invalid education-num:",
      (~df["education-num"].between(1, 16)).sum())
print("Invalid hours-per-week:",
      (df["hours-per-week"] <= 0).sum())

# -----------------------------
# 6. Outlier audit
# -----------------------------
for col in ["age", "fnlwgt", "education-num",
            "capital-gain", "capital-loss", "hours-per-week"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df[col] < lo) | (df[col] > hi)).sum()
    print(col, "IQR outliers:", int(count))

# -----------------------------
# 7. Split before fitted preprocessing
# -----------------------------
target_col = "income"
X = df.drop(columns=[target_col])
y = df[target_col].str.replace(".", "", regex=False).str.strip()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# -----------------------------
# 8. Preprocessing pipeline
# -----------------------------
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns
num_cols = X_train.select_dtypes(exclude=["object", "string"]).columns

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

# -----------------------------
# 9. Final audit
# -----------------------------
print("Remaining training NaNs:",
      np.isnan(X_train_processed.toarray()).sum()
      if hasattr(X_train_processed, "toarray")
      else np.isnan(X_train_processed).sum())


Raw shape: (48842, 15)
Duplicate rows: 29
Invalid ages: 0
Invalid education-num: 0
Invalid hours-per-week: 0
age IQR outliers: 215
fnlwgt IQR outliers: 1453
education-num IQR outliers: 1789
capital-gain IQR outliers: 4035
capital-loss IQR outliers: 2282
hours-per-week IQR outliers: 13489
Processed train shape: (39050, 105)
Processed test shape: (9763, 105)
Remaining training NaNs: 0
